# Baby GPT — Developmental Analysis

Tracks how the model develops over longer training horizons (5m → 15m → 30m → 1h → 2h → 4h).

**Before running:** make sure you have evaluated the timed checkpoints:
```
uv run eval_checkpoints.py
uv run eval_prompts.py model_5m.pt model_15m.pt model_30m.pt model_1h.pt model_2h.pt model_4h.pt
```

In [ ]:
import json
import os
import re
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

HERE = Path(".").resolve()
OUT  = HERE / "output"

LABELS   = ["5m", "15m", "30m", "1h", "2h", "4h"]
LABEL_S  = {"5m": 300, "15m": 900, "30m": 1800, "1h": 3600, "2h": 7200, "4h": 14400}
COLORS   = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#b07aa1"]
COLOR_OF = {l: c for l, c in zip(LABELS, COLORS)}

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

## 1. Checkpoint Inventory

In [ ]:
# Load JSON sidecars for all found checkpoints
rows = []
for label in LABELS:
    pt   = HERE / f"model_{label}.pt"
    meta = HERE / f"model_{label}.json"
    if not pt.exists():
        continue
    d = {"label": label, "pt_size_mb": round(pt.stat().st_size / 1e6, 1)}
    if meta.exists():
        d.update(json.loads(meta.read_text()))
    rows.append(d)

inventory = pd.DataFrame(rows)
print(f"Found {len(inventory)} checkpoint(s)")
display(inventory[[c for c in ["label","elapsed_seconds","num_steps","train_loss_ema","val_bpb","pt_size_mb"] if c in inventory.columns]])

## 2. Developmental Curve — val_bpb & train loss vs time

In [ ]:
# Load eval_results.csv (run eval_checkpoints.py first if missing)
eval_csv = OUT / "eval_results.csv"
if not eval_csv.exists():
    print("eval_results.csv not found — run:  uv run eval_checkpoints.py")
    eval_df = pd.DataFrame()
else:
    eval_df = pd.read_csv(eval_csv)
    eval_df["elapsed_min"] = pd.to_numeric(eval_df["elapsed_seconds"], errors="coerce") / 60
    eval_df["val_bpb"]     = pd.to_numeric(eval_df["val_bpb"], errors="coerce")
    eval_df["train_loss_ema"] = pd.to_numeric(eval_df["train_loss_ema"], errors="coerce")
    display(eval_df)

if not eval_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- val_bpb ---
    ax = axes[0]
    ax.plot(eval_df["elapsed_min"], eval_df["val_bpb"],
            marker="o", linewidth=2, color="#4e79a7", markersize=8)
    for _, row in eval_df.iterrows():
        if pd.notna(row["val_bpb"]):
            ax.annotate(f"{row['val_bpb']:.4f}",
                        (row["elapsed_min"], row["val_bpb"]),
                        textcoords="offset points", xytext=(6, 6), fontsize=9)
    ax.set_xlabel("Elapsed training time (min)")
    ax.set_ylabel("val_bpb (lower is better)")
    ax.set_title("Validation BPB vs Training Time")
    ax.grid(True, alpha=0.2)

    # Marginal improvement annotation
    bpb_vals = eval_df["val_bpb"].dropna().values
    if len(bpb_vals) >= 2:
        total_delta = bpb_vals[0] - bpb_vals[-1]
        ax.set_title(f"Validation BPB vs Training Time  (Δ={total_delta:+.4f} total)")

    # --- train loss EMA ---
    ax2 = axes[1]
    ax2.plot(eval_df["elapsed_min"], eval_df["train_loss_ema"],
             marker="s", linewidth=2, color="#e15759", markersize=8)
    for _, row in eval_df.iterrows():
        if pd.notna(row["train_loss_ema"]):
            ax2.annotate(f"{row['train_loss_ema']:.3f}",
                         (row["elapsed_min"], row["train_loss_ema"]),
                         textcoords="offset points", xytext=(6, 6), fontsize=9)
    ax2.set_xlabel("Elapsed training time (min)")
    ax2.set_ylabel("Train loss EMA (at checkpoint)")
    ax2.set_title("Train Loss EMA vs Training Time")
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(OUT / "developmental_curve.png", bbox_inches="tight")
    plt.show()
    print("Saved output/developmental_curve.png")

## 3. Marginal Returns — How Much Does Each Extra Hour Buy?

In [ ]:
if not eval_df.empty and "val_bpb" in eval_df.columns:
    df = eval_df.dropna(subset=["val_bpb", "elapsed_min"]).copy()
    df["delta_bpb"]  = df["val_bpb"].diff()           # change from previous checkpoint
    df["delta_min"]  = df["elapsed_min"].diff()        # minutes between checkpoints
    df["bpb_per_min"] = df["delta_bpb"] / df["delta_min"]  # improvement rate

    print("Marginal improvement per checkpoint:")
    print(df[["label", "elapsed_min", "val_bpb", "delta_bpb", "bpb_per_min"]].to_string(index=False))

    if len(df) >= 3:
        fig, ax = plt.subplots(figsize=(10, 4))
        valid = df.dropna(subset=["bpb_per_min"])
        bars = ax.bar(valid["label"], valid["delta_bpb"].abs(),
                      color=[COLOR_OF.get(l, "#aaa") for l in valid["label"]])
        ax.set_xlabel("Checkpoint")
        ax.set_ylabel("|Δ val_bpb| (improvement)")
        ax.set_title("Improvement per Training Segment")
        ax.grid(True, alpha=0.2, axis="y")
        for bar, (_, row) in zip(bars, valid.iterrows()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                    f"{row['delta_bpb']:.4f}", ha="center", va="bottom", fontsize=9)
        plt.tight_layout()
        plt.savefig(OUT / "marginal_returns.png", bbox_inches="tight")
        plt.show()

## 4. LR Schedule Shape at Each Duration

Shows how the warmdown curve changes shape as total training time increases, using the current WARMDOWN_RATIO.

In [ ]:
# Read current hyperparams from any available JSON sidecar
warmdown_ratio = 0.70
final_lr_frac  = 0.05
for label in LABELS:
    meta_path = HERE / f"model_{label}.json"
    if meta_path.exists():
        m = json.loads(meta_path.read_text())
        hp = m.get("hyperparams", {})
        warmdown_ratio = hp.get("WARMDOWN_RATIO", warmdown_ratio)
        final_lr_frac  = hp.get("FINAL_LR_FRAC",  final_lr_frac)
        break

def lr_schedule(progress, warmdown_ratio=0.70, final_lr_frac=0.05, warmup_ratio=0.0):
    if progress < warmup_ratio:
        return progress / warmup_ratio if warmup_ratio > 0 else 1.0
    elif progress < 1.0 - warmdown_ratio:
        return 1.0
    else:
        cooldown = (1.0 - progress) / warmdown_ratio
        return cooldown + (1 - cooldown) * final_lr_frac

fig, ax = plt.subplots(figsize=(13, 5))
for label, color in zip(LABELS, COLORS):
    budget_s = LABEL_S[label]
    budget_m = budget_s / 60
    t = np.linspace(0, budget_m, 500)
    progress = t / budget_m
    lr = [lr_schedule(p, warmdown_ratio, final_lr_frac) for p in progress]
    ax.plot(t, lr, label=label, color=color, linewidth=2)

ax.set_xlabel("Elapsed time (min)")
ax.set_ylabel("LR multiplier (fraction of peak)")
ax.set_title(f"LR Schedule Shape  (WARMDOWN_RATIO={warmdown_ratio}, FINAL_LR_FRAC={final_lr_frac})")
ax.legend(title="Budget", loc="upper right")
ax.grid(True, alpha=0.2)
ax.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.savefig(OUT / "lr_schedule.png", bbox_inches="tight")
plt.show()

## 5. Prompt Pack Outputs — Side-by-Side Comparison

In [ ]:
PROMPT_NAMES = [
    "plain_continuation",
    "factual_fragment",
    "longitudinal_anchor",
    "structurally_awkward",
    "anomaly_lure",
    "signature",
    "continuation",
]

def parse_prompt_file(path):
    """Parse output/<label>_prompts.txt into a dict {prompt_name: completion_text}."""
    text = Path(path).read_text(encoding="utf-8")
    blocks = re.split(r"\n\[(", text)
    results = {}
    for i in range(1, len(blocks), 2):
        name_rest = blocks[i]
        name_end  = name_rest.index("]")
        name      = name_rest[:name_end]
        body      = name_rest[name_end+1:]
        comp_match = re.search(r"COMPLETION:\s*(.*)", body, re.DOTALL)
        if comp_match:
            results[name] = comp_match.group(1).strip()
    return results

# Load all available prompt files
prompt_data = {}   # label -> {name -> text}
for label in LABELS:
    p = OUT / f"{label}_prompts.txt"
    if p.exists():
        try:
            prompt_data[label] = parse_prompt_file(p)
        except Exception as e:
            print(f"[{label}] could not parse: {e}")

print(f"Loaded prompt outputs for: {list(prompt_data.keys())}")

In [ ]:
# Display side-by-side for each prompt type
available_labels = [l for l in LABELS if l in prompt_data]

for pname in PROMPT_NAMES:
    md = f"### `{pname}`\n\n"
    has_any = False
    for label in available_labels:
        text = prompt_data[label].get(pname, "*(not found)*")
        # Truncate for readability
        if len(text) > 600:
            text = text[:597] + "..."
        md += f"**{label}**\n> {text}\n\n"
        has_any = True
    if has_any:
        display(Markdown(md + "---"))

## 6. Text Quality Metrics

Automatic proxies for qualitative change. These are rough signals, not ground truth.

In [ ]:
def repetition_onset(text, ngram=4):
    """Word index of the first repeated n-gram. Returns None if no repetition."""
    words = text.lower().split()
    seen = {}
    for i in range(len(words) - ngram + 1):
        g = tuple(words[i:i+ngram])
        if g in seen:
            return i
        seen[g] = i
    return None

def longest_clean_span(text, ngram=4):
    """Number of words before the first repeated n-gram (= repetition_onset, or full length)."""
    onset = repetition_onset(text, ngram)
    words = text.split()
    return onset if onset is not None else len(words)

def repetition_rate(text, ngram=4):
    """Fraction of n-grams that are repeated."""
    words = text.lower().split()
    if len(words) < ngram + 1:
        return 0.0
    grams = [tuple(words[i:i+ngram]) for i in range(len(words)-ngram+1)]
    return 1 - len(set(grams)) / len(grams)

def sentence_completion_rate(text):
    """Fraction of sentence-like units ending with . ! ? — proxy for syntactic tidiness."""
    units = re.split(r"(?<=[.!?])\s+", text.strip())
    if not units:
        return 0.0
    complete = sum(1 for u in units if re.search(r"[.!?]$", u.strip()))
    return complete / len(units)

def type_token_ratio(text):
    """Lexical diversity — length-dependent; use as directional signal only."""
    words = re.findall(r"\b\w+\b", text.lower())
    return len(set(words)) / len(words) if words else 0.0

def avg_sentence_length(text):
    sentences = re.split(r"[.!?]+", text)
    lengths = [len(s.split()) for s in sentences if s.strip()]
    return np.mean(lengths) if lengths else 0.0

# Compute metrics per label per prompt
metric_rows = []
for label in available_labels:
    for pname, text in prompt_data[label].items():
        onset = repetition_onset(text)
        metric_rows.append({
            "label":           label,
            "prompt":          pname,
            "rep_onset_word":  onset if onset is not None else len(text.split()),
            "clean_span":      longest_clean_span(text),
            "rep_rate":        round(repetition_rate(text), 4),
            "sent_completion": round(sentence_completion_rate(text), 3),
            "ttr":             round(type_token_ratio(text), 4),
            "avg_sent_len":    round(avg_sentence_length(text), 1),
        })

if metric_rows:
    mdf = pd.DataFrame(metric_rows)
    summary = mdf.groupby("label")[
        ["rep_onset_word","clean_span","rep_rate","sent_completion","ttr","avg_sent_len"]
    ].mean().round(3)
    summary = summary.reindex([l for l in LABELS if l in summary.index])
    print("Mean across all prompts per checkpoint:")
    display(summary)

    # Plot: the four most diagnostic signals
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    plots = [
        ("clean_span",      "Longest clean span (words)",     "#4e79a7", "higher is better"),
        ("rep_onset_word",  "Repetition onset (word index)",  "#76b7b2", "higher is better"),
        ("rep_rate",        "Repetition rate (4-gram)",       "#e15759", "lower is better"),
        ("sent_completion", "Sentence completion rate",       "#59a14f", "higher is better"),
    ]
    for ax, (col, title, color, note) in zip(axes.flat, plots):
        ax.plot(summary.index, summary[col], marker="o", color=color, linewidth=2, markersize=8)
        for x, y in zip(summary.index, summary[col]):
            ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=9)
        ax.set_title(f"{title}\n({note})", fontsize=10)
        ax.set_xlabel("Checkpoint")
        ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUT / "text_metrics.png", bbox_inches="tight")
    plt.show()
    print("Saved output/text_metrics.png")

## 8. Observations\n\nFill in after reviewing outputs and scoring above.\n\n### Quantitative\n- val_bpb: 5m ___ → 1h ___ → 4h ___\n- Diminishing returns appear after: ___\n- Train loss still falling at 4h? ___\n- Clean span peaks at: ___\n- Sentence completion rate changes noticeably at: ___\n\n### Qualitative\n- Surface fluency (syntax) improves noticeably at: ___\n- Deeper coherence (span, adherence) improves at: ___\n- Repetition loops first appear at: ___ worsen at: ___\n- Weirdness retained through: ___ / collapses at: ___\n- Most interesting output: checkpoint ___, prompt ___\n- Something unexpected: ___\n\n### Next questions\n- ___"

## 7. Qualitative Scoring\n\nRate each checkpoint manually after reading the prompt outputs above.\nScales are defined in `program.md`.\n\n| Prompt | Metric | 5m | 15m | 30m | 1h | 2h | 4h |\n| --- | --- | --- | --- | --- | --- | --- | --- |\n| plain_continuation | coherent span (1–5) | | | | | | |\n| plain_continuation | syntax stability | | | | | | |\n| plain_continuation | prompt adherence | | | | | | |\n| factual_fragment | specificity vs sludge | | | | | | |\n| factual_fragment | prompt adherence | | | | | | |\n| longitudinal_anchor | coherent span (1–5) | | | | | | |\n| longitudinal_anchor | specificity vs sludge | | | | | | |\n| structurally_awkward | syntax stability | | | | | | |\n| structurally_awkward | prompt adherence | | | | | | |\n| anomaly_lure | weirdness retained | | | | | | |\n| anomaly_lure | interestingness | | | | | | |\n| signature | coherent span (1–5) | | | | | | |\n| signature | repetition onset | | | | | | |\n| continuation | coherent span (1–5) | | | | | | |\n| continuation | prompt adherence | | | | | | |\n| **ALL** | **overall interestingness** | | | | | | |"